# 实验：OpenTrack 多智能体 Deep Research Agent

本笔记本实现了基于多智能体的多轮检索。四阶段流水线：
- ScreenAgent → ExecutorAgent → AssessorAgent → SynthesizerAgent
- 配合多路召回 + ReRanker + ConstraintTracker
- 准确率 34%

## 0. 环境与服务说明

In [ ]:
!pip install -r ../core/agent/requirements.txt

## 1. 基础配置

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
sys.path.insert(0, str(Path.cwd()))

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL_NAME = 'qwen_auto'  # 使用 Qwen3-8B
API_KEY = 'dummy'

## 2. 初始化检索、客户端与工具注册表

In [ ]:
from core.agent.vllm_client import VLLMClient
from core.agent.tools import build_searcher, get_agent_tool_specs_and_registry
from core.agent.dataset_utils import load_jsonl
from agent.multiagent import run_agent_loop

bm25_index_path = str(project_root / 'indexes' / 'browsecomp_plus_bm25.sqlite')
hard50_path = str(project_root / 'browsecomp_plus_hard50.jsonl')

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=bm25_index_path)

# 获取工具描述与可执行注册表
tools, registry = get_agent_tool_specs_and_registry(searcher=searcher, k=5, snippet_max_chars=1200)
print('search_type:', searcher.search_type)
print('registered tools:', list(registry.keys()))

## 3. 生成轨迹文件（submission.jsonl）

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from core.agent.vllm_client import VLLMClient
from core.agent.tools import build_searcher, get_agent_tool_specs_and_registry
from agent.multiagent import run_agent_loop


def process_one(row, max_turns=8):
    """每个线程独立创建 client 和 searcher，避免共享状态冲突。"""
    _client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
    _searcher = build_searcher(index_path=bm25_index_path)
    _tools, _registry = get_agent_tool_specs_and_registry(searcher=_searcher)

    answer_text, full_history = run_agent_loop(
        client=_client,
        model=MODEL_NAME,
        query=row["query"],
        tools=_tools,
        registry=_registry,
        max_turns=max_turns,
        max_history_msgs=6
    )
    return {
        "query_id": row["query_id"],
        "status": "completed",
        "predicted_answer": answer_text,
        "messages": full_history,
    }


def generate_submission(rows, output_path: str, max_turns: int = 5, workers: int = 3):
    """并发批量生成轨迹并写入 submission.jsonl。"""
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    records = [None] * len(rows)
    
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_one, row, max_turns): i for i, row in enumerate(rows)}
        for f in as_completed(futures):
            i = futures[f]
            try:
                traj = f.result()
                records[i] = traj
                print(f"[{i+1}/{len(rows)}] query_id={traj['query_id']} done")
            except Exception as e:
                print(f"[{i+1}/{len(rows)}] FAILED: {e}")
                records[i] = {
                    "query_id": rows[i]["query_id"],
                    "status": "error",
                    "predicted_answer": f"ERROR: {e}",
                    "messages": []
                }
    
    with output_file.open("w", encoding="utf-8") as fout:
        for traj in records:
            json.dump(traj, fout, ensure_ascii=False)
            fout.write("\n")
    
    print(f"\nSaved {len(records)} trajectories to: {output_path}")
    return records


submission_path = str(project_root / "runs" / "submission.jsonl")
rows = load_jsonl(hard50_path, limit=50)
records = generate_submission(rows, output_path=submission_path)

print("\n--- 第一条轨迹样例 ---")
sample = records[0]
print(f"query_id: {sample['query_id']}")
print(f"status: {sample['status']}")
print(f"predicted_answer (前200字): {sample['predicted_answer'][:200]}...")
print(f"messages 条数: {len(sample['messages'])}")

## 4. 自动评估

In [ ]:
from core.agent.eval import run_evaluation

# 使用 Qwen3-8B 作为评估模型（也可以换成 openPangu-Embedded-7B-DeepDiver）
EVAL_MODEL = MODEL_NAME

eval_output_path = str("runs/eval_results.jsonl")

summary, details = run_evaluation(
    submission_path=submission_path,
    dataset_path=hard50_path,
    model_name=EVAL_MODEL,
    base_url=VLLM_BASE_URL,
    api_key=API_KEY,
    output_path=eval_output_path,   
    verbose=True,
    temperature=0,
    max_tokens=2000,
    
)

print(f"\n{'='*50}")
print(f"📊 评估摘要")
print(f"{'='*50}")
print(f"总题目数:      {summary['total_queries']}")
print(f"正确数:        {summary['correct']}")
print(f"错误数:        {summary['incorrect']}")
print(f"准确率 (ACC):  {summary['accuracy']:.2%}")
print(f"平均工具调用:  {summary['avg_tool_calls_per_query']}")
print(f"平均检索文档:  {summary['avg_retrieved_docs_per_query']}")
print(f"评估模型:      {summary['eval_model']}")

# 打印几个错误案例
print(f"\n{'='*50}")
print("🔍 错误案例 (前 5 条)")
print(f"{'='*50}")
error_cases = [d for d in details if d["eval_judgment"] == "INCORRECT"]
for case in error_cases[:5]:
    print(f"\nquery_id: {case['query_id']}")
    print(f"  Gold:      {case['gold_answer'][:100]}")
    print(f"  Predicted: {case['predicted_answer'][:100]}")
    print(f"  Reasoning: {case['eval_reasoning'][:150]}")